In [22]:
import re
import os
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [17]:

ytt_api = YouTubeTranscriptApi()
# video_id = "2QPjYmzbz90" 
url = "https://www.youtube.com/watch?v=bxuYDT-BWaI"
video_id = url.split("&")[0].split("=")[1]
 
fetched_transcript = ytt_api.fetch(video_id)

pprint(fetched_transcript[0:5])

[FetchedTranscriptSnippet(text='if you work in Tech or in anything',
                          start=0.0,
                          duration=3.78),
 FetchedTranscriptSnippet(text="adjacent attack you've probably heard",
                          start=2.159,
                          duration=3.841),
 FetchedTranscriptSnippet(text='the abbreviation API being thrown around',
                          start=3.78,
                          duration=4.859),
 FetchedTranscriptSnippet(text="so let's talk about apis what are they",
                          start=6.0,
                          duration=5.96),
 FetchedTranscriptSnippet(text='and why do we need them',
                          start=8.639,
                          duration=3.321)]


In [14]:
fetched_transcript.to_raw_data()[0:10]

[{'text': 'if you work in Tech or in anything',
  'start': 0.0,
  'duration': 3.78},
 {'text': "adjacent attack you've probably heard",
  'start': 2.159,
  'duration': 3.841},
 {'text': 'the abbreviation API being thrown around',
  'start': 3.78,
  'duration': 4.859},
 {'text': "so let's talk about apis what are they",
  'start': 6.0,
  'duration': 5.96},
 {'text': 'and why do we need them', 'start': 8.639, 'duration': 3.321},
 {'text': "let's start with what is an API API",
  'start': 13.38,
  'duration': 5.819},
 {'text': 'stands for application programming',
  'start': 16.98,
  'duration': 4.98},
 {'text': "interface fancy words so let's break it",
  'start': 19.199,
  'duration': 5.16},
 {'text': 'down application in this context just',
  'start': 21.96,
  'duration': 4.44},
 {'text': 'means any software that has a specific',
  'start': 24.359,
  'duration': 4.561}]

In [23]:
def extract_transcript(video_id):
    
    try:
        ytt_api = YouTubeTranscriptApi()
        
        fetched_transcript = ytt_api.fetch(video_id)
        # Merge all text chunks into one single string
        raw_text = " ".join([snippet.text for snippet in fetched_transcript])
        # Remove extra spaces
        transcript_text = re.sub(r"\s+", " ", raw_text).strip()

        return transcript_text
    
    except Exception as e:
        print(f"Error extracting transcript: {e}")
        return "Error extracting transcript"

In [24]:
# url = "https://www.youtube.com/watch?v=b9vk3kHju0I"
url = "https://www.youtube.com/watch?v=bxuYDT-BWaI"
video_id = url.split("&")[0].split("=")[1]
transcript_text = extract_transcript(video_id)
pprint(transcript_text)

("if you work in Tech or in anything adjacent attack you've probably heard the "
 "abbreviation API being thrown around so let's talk about apis what are they "
 "and why do we need them let's start with what is an API API stands for "
 "application programming interface fancy words so let's break it down "
 'application in this context just means any software that has a specific '
 'functionality or purpose interface refers to a contract or a protocol that '
 'dictates how two applications talk to each other using requests and '
 'responses so put together an API is simply a way for different systems or '
 'applications to communicate with each other okay cool in theory so why do we '
 "need apis let's start with a non-technical analogy first let's say you have "
 'a dinner reservation for tonight for three people but you want to change it '
 'to six because some friends decided to join you at the last minute so you '
 "call the restaurant ask them if it's possible to do that and the 

In [25]:
def structure_transcript(raw_transcript):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an assistant that structures raw video transcripts. "
                    "Add proper punctuation, capitalization, and split the text into "
                    "clear sentences and paragraphs. Do not change the meaning or words. "
                    "Return only the structured text."
                ),
            },
            {
                "role": "user",
                "content": raw_transcript,
            },
        ],
    )
    return response.choices[0].message.content

In [26]:
# Test it
url = "https://www.youtube.com/watch?v=bxuYDT-BWaI"
video_id = url.split("&")[0].split("=")[1]
transcript_text = extract_transcript(video_id)
structured = structure_transcript(transcript_text)
print(structured)

If you work in tech or in anything adjacent, you’ve probably heard the abbreviation API being thrown around. So let's talk about APIs: what are they, and why do we need them?

Let's start with what an API is. API stands for Application Programming Interface. Fancy words, so let's break it down. "Application," in this context, just means any software that has a specific functionality or purpose. "Interface" refers to a contract or a protocol that dictates how two applications talk to each other using requests and responses. So, put together, an API is simply a way for different systems or applications to communicate with each other.

Okay, cool in theory. So why do we need APIs? Let's start with a non-technical analogy. First, let's say you have a dinner reservation for tonight for three people, but you want to change it to six because some friends decided to join you at the last minute. You call the restaurant and ask them if it's possible to do that. The customer service person puts y

In [27]:
pprint(structured)

('If you work in tech or in anything adjacent, you’ve probably heard the '
 "abbreviation API being thrown around. So let's talk about APIs: what are "
 'they, and why do we need them?\n'
 '\n'
 "Let's start with what an API is. API stands for Application Programming "
 'Interface. Fancy words, so let\'s break it down. "Application," in this '
 'context, just means any software that has a specific functionality or '
 'purpose. "Interface" refers to a contract or a protocol that dictates how '
 'two applications talk to each other using requests and responses. So, put '
 'together, an API is simply a way for different systems or applications to '
 'communicate with each other.\n'
 '\n'
 "Okay, cool in theory. So why do we need APIs? Let's start with a "
 "non-technical analogy. First, let's say you have a dinner reservation for "
 'tonight for three people, but you want to change it to six because some '
 'friends decided to join you at the last minute. You call the restaurant and '
 "a

In [28]:
def summarize_transcript(transcript):
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an assistant that summarizes video transcripts. "
                        "Write a clear, concise summary in a few paragraphs covering the main points. "
                        "Return only the summary."
                    ),
                },
                {"role": "user", "content": transcript},
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error summarizing transcript: {e}")
        return "Error summarizing transcript"

In [30]:
# summary = summarize_transcript(structured)
pprint(summary)

('APIs, or Application Programming Interfaces, are protocols that enable '
 'different software systems to communicate with each other by sending '
 'requests and receiving responses. They serve as a standardized "contract" '
 'that simplifies complex interactions, allowing applications to access data '
 'or services without needing to understand the internal workings of other '
 "systems. For example, a restaurant's reservation system can be accessed via "
 'an API, allowing customers or other applications to modify reservations '
 'without sharing sensitive internal details.\n'
 '\n'
 'The necessity of APIs becomes clear through analogies like making a '
 'reservation change over the phone—here, the restaurant’s API acts as the '
 'interface that handles requests. In a technical context, web APIs facilitate '
 'data exchange over the internet typically using HTTP requests, which involve '
 'a URL (endpoint), request method, and response containing status codes like '
 "404, indicatin